# Milestone 1 — four CNNs under one protocol
VGG16, ResNet50, MobileNetV2 (width 1.4), and ConvNeXt-Tiny; **20 epochs each: 5 frozen + 15 fine-tuning**, seed 42.

**Colab:** upload this notebook, choose **Runtime → Change runtime type → GPU**, then run cells from top to bottom. A100 matches the original hardware; other CUDA GPUs are supported, subject to memory/runtime limits. Checkpoints save to your own Drive by default. You will be asked to authorize Drive mounting.

**Other Jupyter:** use a CUDA-enabled Linux machine and run the same cells; outputs save locally. Download code uses Unix file APIs. CPU is suitable for inspecting data, but this training cell requires CUDA.

The notebook contains the pipeline code, final IDs and frozen manifests. It downloads only the 7,315 unique images used in classification from the [locked raw dataset](https://drive.google.com/drive/folders/1GKLyP17tm5JC0XENv25GeoGgYynm97LS). All four models share **5,851 train / 735 validation / 729 test** images (stratified 80/10/10 with integer rounding). No shared source files are changed.

This starts a **new training run**, not a display of old results. The best model is selected using validation macro-F1 before test evaluation. Results may differ across GPU/software versions.


## 1. Settings and persistent output


In [ ]:
from pathlib import Path
import importlib.util
IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
# Change RUN_NAME to start an independent run. Keep it to resume Milestone 1.
RUN_NAME = "milestone1_seed42_run1"
SAVE_TO_DRIVE = IN_COLAB
DOWNLOAD_AUTH = "public"  # "adc" uses Google sign-in if public access is unavailable.
BATCH_SIZE = 32  # Original protocol. Try 16 or 8 if your GPU runs out of memory.
PROJECT = (Path("/content") if IN_COLAB else Path.cwd()) / "milestone1_notebook_runtime"
PROJECT.mkdir(parents=True, exist_ok=True)
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_ROOT = Path("/content/drive/MyDrive/AssignmentRuns") / RUN_NAME
else:
    SAVE_ROOT = PROJECT / "saved_runs" / RUN_NAME
SAVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Code and downloaded data:", PROJECT)
print("Persistent checkpoints and results:", SAVE_ROOT)


## 2. Install dependencies
Run installation before importing training libraries. If a package was already imported, restart the kernel and rerun from the top.


In [ ]:
EXTRA_PACKAGES = ["timm==1.0.25"]

import importlib.util, subprocess, sys
packages = ["Pillow==11.3.0", "numpy==2.1.3", "matplotlib==3.10.0",
            "pandas>=2.2,<3", "requests>=2.32,<3", "google-auth>=2,<3",
            "python-docx==1.2.0", "reportlab>=4,<5"]
if importlib.util.find_spec("torch") is None or importlib.util.find_spec("torchvision") is None:
    packages += ["torch>=2.5", "torchvision>=0.20"]
packages += EXTRA_PACKAGES
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *packages], check=True)
print("Dependencies installed. Colab's existing matching torch/CUDA builds were retained.")


## 3. Restore bundled code and frozen split metadata
The large encoded cell is a ZIP of code and metadata. It contains no photos, weights or credentials.


In [ ]:
# Frozen code + metadata packaged inside this notebook; no GitHub clone needed.
import base64, hashlib, io, json, zipfile
from pathlib import Path
BUNDLE_SHA256 = 'd60d7d5012bdd0a616a6067c92fa7577a8fe2658727fa45f6f9fff0b92aeff64'
BUNDLE_B64 = '...'
raw_bundle = base64.b64decode(BUNDLE_B64)
assert hashlib.sha256(raw_bundle).hexdigest() == BUNDLE_SHA256
with zipfile.ZipFile(io.BytesIO(raw_bundle)) as archive:
    for entry in archive.infolist():
        destination = (PROJECT / entry.filename).resolve()
        if PROJECT.resolve() not in destination.parents:
            raise ValueError("Unsafe bundle path")
        content = archive.read(entry)
        destination.parent.mkdir(parents=True, exist_ok=True)
        if destination.exists() and destination.read_bytes() != content:
            raise ValueError(f"Existing code/metadata differs: {destination}. Use a fresh project directory.")
        destination.write_bytes(content)
print(f"Unpacked {len(archive.infolist())} source/metadata files into {PROJECT}")
# You can inspect all extracted .py files from Colab's Files sidebar.


In [ ]:
import hashlib, json, os, shutil, subprocess, sys
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path

sys.path.insert(0, str(PROJECT))
from download_dataset import DriveReader, download_one

def run_script(script, *arguments):
    # Run in a fresh Python process so installed packages and training workers are clean.
    subprocess.run([sys.executable, str(PROJECT / script), *map(str, arguments)],
                   cwd=PROJECT, check=True)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def fetch_expected(folder_id, expected, cache, source_key):
    """Paginated, GET-only downloads; frozen SHA256 verification; safe cache reuse."""
    expected = {str(k): str(v) for k, v in expected.items()}
    cache = Path(cache)
    root = cache / "images"
    pending = {}
    for name, digest in expected.items():
        relative = Path(name)
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError("Unsafe expected image path")
        image = root / relative
        if image.exists():
            if sha256_file(image) != digest:
                raise ValueError(f"Cached image differs from frozen data: {name}")
        else:
            pending[name] = digest
    if not pending:
        print(f"Using {len(expected):,} previously verified images from {root}")
        return root
    if DOWNLOAD_AUTH == "adc":
        if IN_COLAB:
            from google.colab import auth
            auth.authenticate_user()
        else:
            print("Using your existing Google Application Default Credentials.")
    reader = DriveReader(folder_id, DOWNLOAD_AUTH)
    inventory = reader.inventory(workers=4)
    selected = {}
    for item in inventory["files"]:
        name = source_key(item["relative_path"])
        if name in pending:
            if name in selected:
                raise ValueError(f"Ambiguous remote filename: {name}")
            selected[name] = {**item, "relative_path": name}
    if set(selected) != set(pending):
        missing = sorted(set(pending) - set(selected))
        raise ValueError(f"Finalized images are missing from Drive: {missing[:10]}")
    cache.mkdir(parents=True, exist_ok=True)
    done = 0
    with ThreadPoolExecutor(max_workers=4) as pool:
        futures = {pool.submit(download_one, reader, cache, item): name
                   for name, item in selected.items()}
        try:
            for future in as_completed(futures):
                result = future.result()
                name = futures[future]
                if result["sha256"] != expected[name]:
                    raise ValueError(f"Remote bytes differ from frozen experiment: {name}")
                done += 1
                if done % 250 == 0 or done == len(selected):
                    print(f"Verified {done:,}/{len(selected):,} downloads", flush=True)
        except BaseException:
            reader.cancelled.set()
            for future in futures:
                future.cancel()
            raise
    receipt = {"folder_id": folder_id, "expected_files": len(expected),
               "sha256_verified": True, "remote_operations": "GET only",
               "checked_utc": datetime.now(timezone.utc).isoformat()}
    (cache / "download_receipt.json").write_text(json.dumps(receipt, indent=2))
    return root

def link_or_copy(source, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        if sha256_file(destination) != sha256_file(source):
            raise ValueError(f"Existing destination differs: {destination}")
        return
    try:
        os.link(source, destination)
    except OSError:
        shutil.copyfile(source, destination)


## 4. Download and verify the finalized images
Only expected files are fetched. Complete Drive pagination and byte checks prevent silently training on a partial or changed dataset. If public access fails, change DOWNLOAD_AUTH to `adc` and rerun; outside Colab, configure your own Application Default Credentials.


In [ ]:
import csv
PREPARED = PROJECT / "data/prepared-local-v2"
expected = {}
for split in ("train", "val", "test"):
    with (PREPARED / f"{split}.csv").open(newline="") as stream:
        for row in csv.DictReader(stream):
            expected[row["path"]] = row["sha256"]
assert len(expected) == 7315
RAW_ROOT = fetch_expected("1GKLyP17tm5JC0XENv25GeoGgYynm97LS", expected,
                          PROJECT / "raw_cache", lambda name: name)
EXPERIMENT = SAVE_ROOT / "experiment"
EXPERIMENT.mkdir(parents=True, exist_ok=True)
PERSISTENT_PREPARED = EXPERIMENT / "prepared"
PERSISTENT_PREPARED.mkdir(exist_ok=True)
for source in PREPARED.iterdir():
    if source.is_file():
        link_or_copy(source, PERSISTENT_PREPARED / source.name)
print("All finalized split images verified. Original raw data remains untouched.")


## 5. Inspect preprocessing and final IDs
Duplicates were removed before splitting. Training uses EXIF-oriented RGB, random 224-pixel crops, horizontal flips and color jitter; evaluation uses resize-short-side 256 and centre-crop 224. Both use ImageNet normalization. The frozen audit records exclusions and reasons.


In [ ]:
import pandas as pd
from IPython.display import display
mapping = json.loads((PREPARED / "class_mapping.json").read_text())
audit = json.loads((PREPARED / "audit.json").read_text())
print("Split counts:", audit["split_counts"], "Seed:", audit["seed"])
display(pd.DataFrame(mapping))
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
rows = list(csv.DictReader((PREPARED / "train.csv").open()))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for axis, row in zip(axes.flat, rows[::max(1, len(rows)//8)][:8]):
    with Image.open(RAW_ROOT / row["path"]) as image:
        axis.imshow(ImageOps.exif_transpose(image).convert("RGB"))
    axis.set_title(f"{row['object_id']}: {row['object_name']}", fontsize=9)
    axis.axis("off")
plt.tight_layout(); plt.show()


## 6. Train all four models and evaluate
This is the long GPU cell. Every model completes all 20 epochs; there is no early stopping. `best.pt` and `last.pt` are saved every epoch in the persistent output. Rerun with the same RUN_NAME/settings after an interruption to resume Milestone 1. Changing settings requires a new RUN_NAME.


In [ ]:
import torch
assert torch.cuda.is_available(), "Enable a GPU runtime before training."
print(torch.cuda.get_device_name(0), "torch", torch.__version__)
TRAIN_OUTPUT = EXPERIMENT / "benchmark_run"
arguments = ["--raw-root", RAW_ROOT, "--prepared-dir", PERSISTENT_PREPARED,
             "--output", TRAIN_OUTPUT, "--epochs", 20, "--frozen-epochs", 5,
             "--seed", 42, "--batch-size", BATCH_SIZE, "--workers", 2, "--device", "cuda"]
if (TRAIN_OUTPUT / "config.json").exists():
    arguments += ["--resume"]
run_script("benchmark/train_benchmark.py", *arguments)
display(pd.read_csv(TRAIN_OUTPUT / "comparison.csv"))
print("Validation selection:", json.loads((TRAIN_OUTPUT / "selection.json").read_text())["selected_model"])


## 7. Generate plots and an editable Word report
These files are built from this run's predictions and metrics. The report is a generated draft: add group details and review its wording/layout before submitting.


In [ ]:
REPORTS = PROJECT / "reports/group_benchmark"
run_script("benchmark/analyze_benchmark.py", "--experiment", EXPERIMENT,
           "--raw-root", RAW_ROOT, "--output", REPORTS)
(PROJECT / "tmp/docx").mkdir(parents=True, exist_ok=True)
run_script("benchmark/render_word_report.py")
SAVED_REPORTS = SAVE_ROOT / "report"
shutil.copytree(REPORTS, SAVED_REPORTS, dirs_exist_ok=True)
docx_source = PROJECT / "output/docx/milestone1_group_comparison.docx"
shutil.copyfile(docx_source, SAVED_REPORTS / docx_source.name)
from IPython.display import Image as DisplayImage
display(DisplayImage(filename=str(SAVED_REPORTS / "plots/comparison.png")))
display(DisplayImage(filename=str(SAVED_REPORTS / "plots/learning_curves.png")))
print("Word report:", SAVED_REPORTS / docx_source.name)
print("Model checkpoints:", TRAIN_OUTPUT)


## 8. Classify a new image with this run's selected model
Optional: put an image in the runtime, then set IMAGE_PATH. Output IDs/names come from the checkpoint's final mapping.


In [ ]:
IMAGE_PATH = ""  # e.g. /content/my_photo.jpg; blank skips this cell.
if IMAGE_PATH:
    from benchmark.train_benchmark import Classifier
    from train import load_checkpoint, MEAN, STD
    from torchvision import transforms
    name = json.loads((TRAIN_OUTPUT / "selection.json").read_text())["selected_model"]
    checkpoint = load_checkpoint(TRAIN_OUTPUT / name / "best.pt")
    model = Classifier(name, len(checkpoint["classes"]), pretrained=False).eval().to("cuda")
    model.load_state_dict(checkpoint["model"])
    transform = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224),
                                    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
    with Image.open(IMAGE_PATH) as image:
        rgb = ImageOps.exif_transpose(image).convert("RGB")
    with torch.inference_mode():
        probabilities = model(transform(rgb).unsqueeze(0).to("cuda")).softmax(1)[0]
    values, indices = probabilities.topk(5)
    display(pd.DataFrame([{**checkpoint["classes"][i], "score": float(v)}
                          for v, i in zip(values.cpu(), indices.cpu().tolist())]))
    display(rgb)
else:
    print("Set IMAGE_PATH to run an optional classification demo.")


Your checkpoint directories include each model's `best.pt` and `last.pt`. Download the report using the Colab Files sidebar, or open it in the AssignmentRuns folder in your own Drive. Do not publish credentials or modify the locked shared dataset.
